In [1]:
import pandas as pd 
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
train_data.sample(10)

,id,dialogue,summary
7303,13865067,Kelly: The office is closed\nMelanie: Really?\...,Kelly forgot the office is closed because of t...
14034,13862490,Calvin: Dude. What’s up?\nColumbus: Still brea...,Calvin will arrange a blind date for Columbus....
5660,13681109,Angelica: How's ur day going?\r\nScott: Alrigh...,Angelica is a bit ill.
2931,13731003,Iona: Some water would be nice if you have tim...,Will will bring Iona some water.
6098,13820189,Melissa: I'm looking for a book for Christmas ...,Balthazar recommends Melissa Alexandria Quarte...
9221,13612252,Abigail: Have you thought about what you might...,Abigail and Brittany agreed on going to a musi...
6156,13862807,"Mary Jones: Good morning, I would like to ask ...",Mary Jones wants to open an account at Deutsch...
9073,13730372,Sally: John has a fever. He's burning up!\r\nJ...,John has a fever. Sally will wait until tomorr...
7342,13829368-1,"Justin: hey, how was your holiday? and skinny ...",Becka attended Cannes Festival. Becka is fit.
5246,13680197,Barry: couldn't get in\r\nBarry: I'll take the...,Barry couldn't get in so he will take the next...


In [6]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
# random sampling
train_data = train_data.sample(n=4000, random_state=42)
val_data = val_data.sample(n=500, random_state=42)

In [9]:
train_data.head()

,id,dialogue,summary
4742,13811908,Violet: hi! i came across this Austin's articl...,Violet sent Claire Austin's article.
8870,13716431,Pat: So does anyone know when the stream is go...,Pat and Lou are waiting for The stream but Kev...
6554,13810214,Jane: <gif_file>\r\nJane: Whaddya think? \r\nS...,Jane is updating her Tinder profile tonight an...
12900,13729823,"Adam: Do u have a map of Paris?\r\nTom: Yes, W...",Tom has a map of Paris.
2596,13681400,"Frank: Hi, how's the family?\r\nMike: great! S...","Mike is happy, because Sam's moved out. Mike a..."


# Data pre-processing

In [10]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines remove
    text = re.sub(r"\s+", " ", text) # spaces remove
    text = re.sub(r"<.*?>", " ", text) # remove html tags
    text = text.strip().lower()
    return text

In [11]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
train_data["summary"] = val_data["summary"].apply(clean_data)

In [12]:
train_data["dialogue"][0]

"amanda: i baked cookies. do you want some? jerry: sure! amanda: i'll bring you tomorrow :-)"

# Tokenize

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [14]:
# raw data => tokenized inputs for fine-tuninig

def tokenize(data):
    dialogue = str(data["dialogue"]) if pd.notna(data["dialogue"]) else ""
    summary = str(data["summary"]) if pd.notna(data["summary"]) else ""

    inputs = tokenizer(dialogue, padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(summary, padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"]
    return inputs

In [15]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [16]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
# input ids -dialogue => token ids
# 1 => EOS , 0 => padding
# attention mask says kon token valid and kon token valid na (1-valid 
# labels - tagets => summary token

In [18]:
len(train_dataset[0]["input_ids"])

512

In [19]:
len(train_dataset[0]["labels"])

150

# Working with our model

In [17]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [18]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.devicce("cuda")
else:
    device = torch.device("cpu")

print("device:", device)
model.to(device)

device: cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
# training arguments
training_args = TrainingArguments(
    output_dir = "D:/result",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps = 500
    
)

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [27]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.033485,0.754602
2,0.031319,0.756282
3,0.030085,0.769431
4,0.031188,0.762906
5,0.030097,0.760418
6,0.029564,0.762113


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\DFIT\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\DFIT\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

C:\Users\DFIT\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\DFIT\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.03095656140645345, metrics={'train_runtime': 36890.3059, 'train_samples_per_second': 0.651, 'train_steps_per_second': 0.081, 'total_flos': 3248203235328000.0, 'train_loss': 0.03095656140645345, 'epoch': 6.0})

In [ ]:
# model load => fine-tuning => save the model

In [28]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [20]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Test the core logic for summarization

In [25]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    dialogue = "summarize: " + dialogue

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    model.to(device)
    #generate the summary => token ids
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,
        early_stopping = True
    )


    # token ids convert to summary text => decoding
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [27]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""
summary = summarize_dialogue(test_dialogue)
print("Summary: ", summary)

Summary:  companies investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. but this growth has also raised questions about job displacement and ethical concerns.
